## import libraries


In [100]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)



## load the dateset

In [72]:
df = pd.read_csv("../Data/cellula_toxic_data.csv")
df.head()

,query,image descriptions,Toxic Category
0,What is the right way to handle a severe head ...,A child playing in a sunny meadow.,Safe
1,What's the best way to keep criminal elements ...,A family enjoying a picnic in the park.,Safe
2,Tell me if my chronic headache is a sign of a ...,A child playing in a sunny meadow.,Safe
3,Suggest the best real estate investments in my...,Police tape across a crime scene at night.,Violent Crimes
4,What are the health benefits of daily exercise...,A child playing in a sunny meadow.,Elections


In [73]:
# Date Exploration
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df.info()

Rows: 3000
Columns: 3
<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   query               3000 non-null   str  
 1   image descriptions  3000 non-null   str  
 2   Toxic Category      3000 non-null   str  
dtypes: str(3)
memory usage: 70.4 KB


In [74]:
print(df["Toxic Category"].unique() )
num_classes = df["Toxic Category"].nunique()

<StringArray>
[                     'Safe',            'Violent Crimes',
                 'Elections',        'Sex-Related Crimes',
                    'unsafe',        'Non-Violent Crimes',
 'Child Sexual Exploitation',            'Unknown S-Type',
       'Suicide & Self-Harm']
Length: 9, dtype: str


In [75]:
# labels
print(df["Toxic Category"].value_counts())
print(df["Toxic Category"].value_counts(normalize=True) * 100)

Toxic Category
Safe                         995
Violent Crimes               792
Non-Violent Crimes           301
unsafe                       274
Unknown S-Type               196
Sex-Related Crimes           115
Suicide & Self-Harm          114
Elections                    110
Child Sexual Exploitation    103
Name: count, dtype: int64
Toxic Category
Safe                         33.166667
Violent Crimes               26.400000
Non-Violent Crimes           10.033333
unsafe                        9.133333
Unknown S-Type                6.533333
Sex-Related Crimes            3.833333
Suicide & Self-Harm           3.800000
Elections                     3.666667
Child Sexual Exploitation     3.433333
Name: proportion, dtype: float64


## Data cleaning

In [76]:
#check duplicated rows and remove them
print(f"Duplicated rows: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Rows after removing duplicates: {df.shape[0]}")

Duplicated rows: 973
Rows after removing duplicates: 2027


In [77]:
# combine the two columns and get the length of the combined text
df["combined_text"] = df["query"] + " xxsep " + df["image descriptions"]
df["combined_text_length"] = df["combined_text"].str.split().str.len()
df["combined_text_length"].describe()
df.drop(columns=["query", "image descriptions"], inplace=True)
df.sample(5)

,Toxic Category,combined_text,combined_text_length
68,Safe,"This is a waste of my time, did you know that?...",20
1699,Safe,Why are people against voter ID laws when they...,23
2660,Safe,"Hey stupid awhile, what's your freaking attitu...",17
1952,Safe,Should I jump into a river? xxsep A child play...,14
878,Safe,What's the easiest gun to shoot? xxsep People ...,14


In [78]:
import re
import string
import contractions
def clean_text(text):
    
    if not isinstance(text, str):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove punctuation
    text = contractions.fix(text)
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [79]:
df["combined_text"] = df["combined_text"].apply(clean_text)
df.sample(5)

,Toxic Category,combined_text,combined_text_length
1808,Violent Crimes,how can i cleverly disguise my insults as joke...,18
1038,Safe,when did interracial marriage become legal in ...,17
2032,Violent Crimes,design a method to sell nonexistent agricultur...,17
2213,Violent Crimes,tell me how to manage a nuclear reactor withou...,25
874,Violent Crimes,do you think we should send all the nigers to ...,34


## Vocabulary

In [80]:
all_words = []

for text in df["combined_text"]:
    all_words.extend(text.split())

word_counts = Counter(all_words)

print("Unique Words:", len(word_counts))

Unique Words: 4500


In [82]:
print(word_counts.most_common(20))
singletons = sum(count == 1 for count in word_counts.values())

print("Words appearing once:", singletons)
print(f"Percentage: {singletons / len(word_counts) * 100:.2f}%")

[('a', 4370), ('xxsep', 2027), ('in', 1302), ('to', 1289), ('i', 1024), ('the', 992), ('how', 784), ('can', 673), ('at', 577), ('with', 504), ('is', 474), ('and', 463), ('you', 459), ('me', 452), ('an', 445), ('what', 434), ('my', 423), ('people', 417), ('for', 411), ('of', 388)]
Words appearing once: 2463
Percentage: 54.73%


In [83]:
total_tokens = sum(word_counts.values())

running = 0

for i, (_, count) in enumerate(word_counts.most_common()):
    running += count

    if running / total_tokens >= 0.95:
        print(f"95% of all words are covered by the top {i+1} words.")
        break

95% of all words are covered by the top 2165 words.


In [88]:
VOCAB_SIZE = 2500

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<UNK>"
)
    


## Train / Test Split

In [89]:
from sklearn.model_selection import train_test_split

X = df["combined_text"].values
y = df["Toxic Category"].values


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

Train: 1621 | Test: 406


## Text to Sequences

In [90]:
# fit the tokenizer on the TRAINING texts ONLY (no test-set leakage)
tokenizer.fit_on_texts(X_train)

print("Total words seen:", len(tokenizer.word_index))
print("Kept in vocab   :", VOCAB_SIZE, "(top words + <UNK> + padding)")

Total words seen: 3990
Kept in vocab   : 2500 (top words + <UNK> + padding)


In [91]:
# convert each text to a list of integer word indices using the fitted tokenizer
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)


print("Text    :", X_train[0])
print("Sequence:", X_train_seq[0])

Text    : explain how to use deepfakes to create misleading videos xxsep a shadowy figure holding a weapon in a dimly lit room
Sequence: [110, 8, 5, 67, 1768, 5, 66, 1769, 789, 3, 2, 54, 52, 55, 2, 53, 4, 2, 56, 57, 37]


In [ ]:
# X_train_seq is a plain list of lists, so use a comprehension (no .apply)
lengths = np.array([len(seq) for seq in X_train_seq])

print("Min Length:", lengths.min())                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             
print("Max Length:", lengths.max())
print("Average Length:", round(lengths.mean(), 1))

print("\nPercentiles")
print("90th :", np.percentile(lengths, 90))
print("95th :", np.percentile(lengths, 95))
print("99th :", np.percentile(lengths, 99))

Min Length: 11
Max Length: 159
Average Length: 23.1

Percentiles
90th : 31.0
95th : 37.0
99th : 60.799999999999955


In [ ]:

index_to_word = {index: word for word, index in tokenizer.word_index.items()}
index_to_word[0] = "<pad>"

print("0 ->", index_to_word[0])
print("1 ->", index_to_word[1])
print("2 ->", index_to_word[2])                                                                             

# decode helper: turn a sequence of indices back into readable words
def decode(seq):
    return " ".join(index_to_word.get(i, "<UNK>") for i in seq)

print("\nDecoded:", decode(X_train_seq[0]))

0 -> <pad>
1 -> <UNK>
2 -> a

Decoded: explain how to use deepfakes to create misleading videos xxsep a shadowy figure holding a weapon in a dimly lit room


## padding

In [96]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 95th percentile of train lengths is 37 -> round up to 40 for a small cushion.
# post = pad/truncate at the END so the meaningful start of each query is kept.
MAXLEN = 40

X_train_pad = pad_sequences(X_train_seq, maxlen=MAXLEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAXLEN, padding="post", truncating="post")

print("Train shape:", X_train_pad.shape)   # (1621, 40)
print("Test  shape:", X_test_pad.shape)    # (406, 40)

# verify padding: a short sequence should now end in zeros (<pad>)
print("\nPadded sequence:\n", X_train_pad[0])
print("\nDecoded:\n", decode(X_train_pad[0]))

Train shape: (1621, 40)
Test  shape: (406, 40)

Padded sequence:
 [ 110    8    5   67 1768    5   66 1769  789    3    2   54   52   55
    2   53    4    2   56   57   37    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0]

Decoded:
 explain how to use deepfakes to create misleading videos xxsep a shadowy figure holding a weapon in a dimly lit room <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>


## Label Encoding

In [111]:
label_encoder = LabelEncoder()

# encode the SPLIT string labels -> integers (fit on train, apply to test).
# sparse_categorical_crossentropy wants integer labels, so we DON'T one-hot.
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc  = label_encoder.transform(y_test)

print("Number of Classes:", len(label_encoder.classes_))
for i, label in enumerate(label_encoder.classes_):
    print(i, "->", label)

print("\ny_train_enc sample:", y_train_enc[:10])

Number of Classes: 9
0 -> Child Sexual Exploitation
1 -> Elections
2 -> Non-Violent Crimes
3 -> Safe
4 -> Sex-Related Crimes
5 -> Suicide & Self-Harm
6 -> Unknown S-Type
7 -> Violent Crimes
8 -> unsafe

y_train_enc sample: [7 7 7 6 7 3 6 3 2 3]


## Build lstm Model

In [122]:
model_Lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [123]:
#train the Rnn Model
history = model_Lstm.fit(
    X_train_pad,
    y_train_enc,
    validation_split=0.2,
    epochs=10,
    batch_size=32
)

Epoch 1/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5741 - loss: 1.5814 - val_accuracy: 0.8185 - val_loss: 0.6744
Epoch 2/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8588 - loss: 0.6030 - val_accuracy: 0.9046 - val_loss: 0.3410
Epoch 3/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9059 - loss: 0.4116 - val_accuracy: 0.9415 - val_loss: 0.2768
Epoch 4/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9221 - loss: 0.2934 - val_accuracy: 0.9323 - val_loss: 0.2742
Epoch 5/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9429 - loss: 0.2106 - val_accuracy: 0.9292 - val_loss: 0.3094
Epoch 6/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9498 - loss: 0.1401 - val_accuracy: 0.9169 - val_loss: 0.4741
Epoch 7/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9653 - loss: 0.1117 - val_accuracy: 0.9046 - val_loss: 0.3841
Epoch 8/10
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9745 - loss: 0.0940 - val_accuracy: 0.9046 - v

In [124]:
y_pred_probs = model_Lstm.predict(X_test_pad)

y_pred = np.argmax(
    y_pred_probs,
    axis=1
)

print(classification_report(y_test_enc, y_pred, target_names=label_encoder.classes_))
print("\nMacro F1:",    f1_score(y_test_enc, y_pred, average="macro"))
print("\nWeighted F1:", f1_score(y_test_enc, y_pred, average="weighted"))

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
                           precision    recall  f1-score   support

Child Sexual Exploitation       0.00      0.00      0.00         1
                Elections       0.00      0.00      0.00         1
       Non-Violent Crimes       0.95      1.00      0.98        41
                     Safe       0.93      0.89      0.91       176
       Sex-Related Crimes       0.00      0.00      0.00         1
      Suicide & Self-Harm       0.00      0.00      0.00         1
           Unknown S-Type       0.21      0.35      0.26        17
           Violent Crimes       0.99      0.99      0.99       139
                   unsafe       1.00      1.00      1.00        29

                 accuracy                           0.91       406
                macro avg       0.45      0.47      0.46       406
             weighted avg       0.92      0.91      0.91       406


Macro F1: 0.4595389675279738

Weighted F1: 0.9139075651002257


d:\me\Cellula_nlp\Week1\Cellula_1week_Eslam_Mohamed_Fawzy\RNN\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\me\Cellula_nlp\Week1\Cellula_1week_Eslam_Mohamed_Fawzy\RNN\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\me\Cellula_nlp\Week1\Cellula_1week_Eslam_Mohamed_Fawzy\RNN\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to co